# 🔍 YouTube Trending Videos - Exploratory Data Analysis

**Project:** YZV475E Data Visualization Term Project  
**Team:** EnAi (Muhammed Abdullah Özdemir, Hikmet Gültekin)

---

## Notebook Goals
1. Deep dive into the data to find interesting patterns
2. Discover potential "story" angles for the dashboard
3. Identify which visualizations will be most impactful
4. Generate insights for the presentation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries imported")

In [ ]:
# Load the cleaned data
OUTPUT_PATH = Path("../output")
df = pd.read_csv(OUTPUT_PATH / 'youtube_trending_cleaned.csv')

# Convert dates back to datetime
df['publishedAt'] = pd.to_datetime(df['publishedAt'])
df['trending_date'] = pd.to_datetime(df['trending_date'])

print(f"✅ Loaded {len(df):,} rows")

---
## 🌍 1. Geographic Distribution Analysis

In [ ]:
# Videos per country
country_stats = df.groupby(['country', 'country_name']).agg({
    'video_id': 'count',
    'view_count': ['mean', 'sum'],
    'likes': 'mean',
    'engagement_rate': 'mean'
}).round(2)

country_stats.columns = ['video_count', 'avg_views', 'total_views', 'avg_likes', 'avg_engagement']
country_stats = country_stats.sort_values('video_count', ascending=False)
country_stats

In [ ]:
# Visualization: Country comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Video count
country_stats['video_count'].plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Trending Videos by Country')
axes[0].set_xlabel('Number of Videos')

# Average engagement
country_stats['avg_engagement'].plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Average Engagement Rate by Country')
axes[1].set_xlabel('Engagement Rate')

plt.tight_layout()
plt.show()

print("💡 INSIGHT: Which countries have the most videos? Which have highest engagement?")

---
## 🏷️ 2. Category Analysis

In [ ]:
# Global category distribution
category_dist = df['category_name'].value_counts()
category_pct = (category_dist / len(df) * 100).round(1)

print("🏷️ Category Distribution (Global):")
for cat, count in category_dist.items():
    print(f"   {cat:25}: {count:>8,} ({category_pct[cat]:>5.1f}%)")

In [ ]:
# Category preferences by country - THIS IS GOLD FOR CULTURAL INSIGHTS
category_by_country = pd.crosstab(df['country_name'], df['category_name'], normalize='index') * 100

# Heatmap
plt.figure(figsize=(16, 8))
sns.heatmap(category_by_country, annot=True, fmt='.1f', cmap='YlOrRd', 
            linewidths=0.5, cbar_kws={'label': 'Percentage'})
plt.title('Category Preferences by Country (% of trending videos)', fontsize=14)
plt.xlabel('Category')
plt.ylabel('Country')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("💡 INSIGHT: Look for countries with unusual category preferences!")

In [ ]:
# Find distinctive categories per country
global_avg = df['category_name'].value_counts(normalize=True) * 100

print("🔍 Categories that OVER-INDEX in each country:")
for country in df['country_name'].unique():
    country_data = df[df['country_name'] == country]
    country_dist = country_data['category_name'].value_counts(normalize=True) * 100
    
    # Find categories where country is 2x+ global average
    diff = country_dist - global_avg
    top_diff = diff.sort_values(ascending=False).head(2)
    
    distinctive = [f"{cat} (+{val:.1f}%)" for cat, val in top_diff.items() if val > 3]
    if distinctive:
        print(f"   {country}: {', '.join(distinctive)}")

---
## 📅 3. Time Series Analysis

In [ ]:
# Monthly trending volume
monthly = df.groupby('trending_year_month').size().reset_index(name='count')
monthly['date'] = pd.to_datetime(monthly['trending_year_month'])
monthly = monthly.sort_values('date')

plt.figure(figsize=(14, 5))
plt.plot(monthly['date'], monthly['count'], marker='o', markersize=3, linewidth=1)
plt.title('Monthly Trending Video Count (All Countries)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Number of Trending Videos')

# Mark COVID start
covid_start = pd.Timestamp('2020-03-01')
if monthly['date'].min() < covid_start < monthly['date'].max():
    plt.axvline(x=covid_start, color='red', linestyle='--', alpha=0.7, label='COVID-19 Pandemic Start')
    plt.legend()

plt.tight_layout()
plt.show()

print("💡 INSIGHT: Any noticeable trends? COVID impact? Seasonal patterns?")

In [ ]:
# Category trends over time - THE PANDEMIC STORY
category_time = df.groupby(['trending_year_month', 'category_name']).size().reset_index(name='count')
category_time['date'] = pd.to_datetime(category_time['trending_year_month'])

# Focus on key categories that might show COVID impact
key_categories = ['Gaming', 'Entertainment', 'News & Politics', 'Education', 'Music', 'Sports']

plt.figure(figsize=(14, 6))
for cat in key_categories:
    cat_data = category_time[category_time['category_name'] == cat].sort_values('date')
    plt.plot(cat_data['date'], cat_data['count'], label=cat, linewidth=2)

plt.axvline(x=pd.Timestamp('2020-03-01'), color='gray', linestyle='--', alpha=0.5, label='COVID Start')
plt.title('Category Trends Over Time', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Number of Videos')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("💡 INSIGHT: Did Gaming/Education spike during COVID? Did Sports drop?")

In [ ]:
# Pre-COVID vs Post-COVID category comparison
df['covid_era'] = df['trending_date'].apply(lambda x: 'Post-COVID' if x >= pd.Timestamp('2020-03-15') else 'Pre-COVID')

covid_comparison = pd.crosstab(df['covid_era'], df['category_name'], normalize='index') * 100

# Calculate change
if 'Pre-COVID' in covid_comparison.index and 'Post-COVID' in covid_comparison.index:
    change = covid_comparison.loc['Post-COVID'] - covid_comparison.loc['Pre-COVID']
    change = change.sort_values(ascending=False)
    
    plt.figure(figsize=(12, 6))
    colors = ['green' if x > 0 else 'red' for x in change.values]
    change.plot(kind='barh', color=colors)
    plt.title('Category Share Change: Pre-COVID vs Post-COVID', fontsize=14)
    plt.xlabel('Change in Percentage Points')
    plt.axvline(x=0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.show()
    
    print("💡 INSIGHT: This shows which categories gained/lost during pandemic!")

---
## 📊 4. Engagement Analysis

In [ ]:
# Views distribution (log scale makes more sense)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# View count distribution
axes[0].hist(np.log10(df['view_count'] + 1), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('View Count Distribution (log10 scale)')
axes[0].set_xlabel('log10(View Count)')
axes[0].set_ylabel('Frequency')

# Engagement rate distribution
axes[1].hist(df['engagement_rate'].dropna(), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Engagement Rate Distribution')
axes[1].set_xlabel('Engagement Rate')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Views vs Likes scatter (sample for performance)
sample = df.sample(n=min(50000, len(df)), random_state=42)

plt.figure(figsize=(12, 6))
plt.scatter(sample['view_count'], sample['likes'], alpha=0.1, s=1)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('View Count (log scale)')
plt.ylabel('Likes (log scale)')
plt.title('Views vs Likes Relationship')
plt.tight_layout()
plt.show()

# Correlation
corr = df[['view_count', 'likes', 'comment_count']].corr()
print("📊 Correlation Matrix:")
print(corr.round(3))

In [ ]:
# Engagement by category
cat_engagement = df.groupby('category_name').agg({
    'view_count': 'mean',
    'likes': 'mean',
    'engagement_rate': 'mean',
    'video_id': 'count'
}).round(2)
cat_engagement.columns = ['avg_views', 'avg_likes', 'avg_engagement', 'video_count']
cat_engagement = cat_engagement.sort_values('avg_views', ascending=False)

print("🏷️ Category Engagement Metrics:")
cat_engagement

---
## ⏰ 5. Publishing Patterns

In [ ]:
# Best hour to publish
hourly = df.groupby('publish_hour').agg({
    'video_id': 'count',
    'view_count': 'mean'
}).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(hourly['publish_hour'], hourly['video_id'], color='steelblue')
axes[0].set_title('Videos Published by Hour (UTC)')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Number of Videos')

axes[1].bar(hourly['publish_hour'], hourly['view_count'], color='coral')
axes[1].set_title('Average Views by Publish Hour')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Average Views')

plt.tight_layout()
plt.show()

print("💡 INSIGHT: Is there a 'best time' to publish for trending?")

In [ ]:
# Day of week patterns
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['publish_day_of_week'] = pd.Categorical(df['publish_day_of_week'], categories=dow_order, ordered=True)

dow_stats = df.groupby('publish_day_of_week').agg({
    'video_id': 'count',
    'view_count': 'mean'
}).reindex(dow_order)

fig, ax = plt.subplots(figsize=(10, 5))
dow_stats['video_id'].plot(kind='bar', color='steelblue', ax=ax)
ax.set_title('Videos Published by Day of Week')
ax.set_xlabel('Day')
ax.set_ylabel('Number of Videos')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 🏆 6. Top Performers Analysis

In [ ]:
# Top channels by trending appearances
top_channels = df.groupby('channelTitle').agg({
    'video_id': 'count',
    'view_count': 'sum',
    'country': lambda x: ', '.join(x.unique()[:3])  # Countries they trend in
}).sort_values('video_id', ascending=False).head(20)

top_channels.columns = ['trending_count', 'total_views', 'countries']
print("🏆 Top 20 Channels by Trending Appearances:")
top_channels

In [ ]:
# Most viewed videos
top_videos = df.nlargest(20, 'view_count')[['title', 'channelTitle', 'view_count', 'category_name', 'country']]
print("🎬 Top 20 Most Viewed Videos:")
top_videos

In [ ]:
# Videos that trended in multiple countries (global hits)
video_countries = df.groupby('video_id').agg({
    'country': 'nunique',
    'title': 'first',
    'channelTitle': 'first',
    'category_name': 'first',
    'view_count': 'max'
}).reset_index()

global_hits = video_countries[video_countries['country'] >= 5].sort_values('country', ascending=False)
print(f"🌍 Videos that trended in 5+ countries: {len(global_hits)}")
print("\nTop Global Hits:")
global_hits.head(15)

---
## 🏷️ 7. Tag Analysis

In [ ]:
# Extract and count tags (sample for speed)
sample_tags = df.sample(n=min(100000, len(df)), random_state=42)

all_tags = []
for tags in sample_tags['tags'].dropna():
    if tags != '[None]':
        for tag in str(tags).split('|'):
            tag = tag.strip().lower()
            if tag and len(tag) > 2:
                all_tags.append(tag)

tag_counts = pd.Series(all_tags).value_counts().head(30)

plt.figure(figsize=(12, 8))
tag_counts.plot(kind='barh', color='steelblue')
plt.title('Top 30 Most Common Tags')
plt.xlabel('Frequency')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
## 📝 8. Story Ideas Summary

Run all cells above and document your findings here:

In [ ]:
print("="*60)
print("📝 POTENTIAL STORY ANGLES")
print("="*60)
print("""
Based on the EDA, potential storytelling angles:

1. 🦠 THE PANDEMIC EFFECT
   - How COVID-19 changed video consumption globally
   - Category shifts (Gaming up? Sports down?)
   - Country-specific lockdown impacts

2. 🌍 CULTURAL FINGERPRINTS
   - Each country's unique content preferences
   - What makes each market different?
   - Global vs local content performance

3. 🎯 THE VIRAL FORMULA
   - What makes videos trend?
   - Optimal publish times
   - Tag strategies that work
   - Cross-country viral hits

4. 📈 PLATFORM EVOLUTION
   - How YouTube trending has changed 2020-2023
   - Rise of new content types
   - Channel dynamics

DOCUMENT YOUR KEY FINDINGS BELOW:
""")

# Add your observations here after running the notebook:
findings = """
[ Your key findings go here ]
"""
print(findings)